**The German International University of Applied Sciences**

**Faculty of Informatics and Computer Science**

**Data Engineering and visualisation**

**Project 1**

---

This part of the project is my individual contribution. I asked the following two research questions:

1. How does affordable housing production vary throughout the year?

2. Do housing completions precede changes in traffic harm?

The first thing to do is to import all needed libraries and load the data sets into data frames.

In [11]:
import pandas as pd
import plotly.express as px

df_housing = pd.read_csv("datasets/Housing_Cleaned.csv")
df_transport = pd.read_csv("datasets/Transportation_Cleaned.csv")

---

### Affordable Housing Throughout The Year

I will start by counting the number of housing units produced in each month, then I will display the data as a bar chart. I will only look at data from the past five years.

In [19]:
df_housing["project_start_date"] = pd.to_datetime(
    df_housing["project_start_date"],
    errors="coerce"
)

latest_date = df_housing["project_start_date"].max()
start_date = latest_date - pd.DateOffset(years=5)

housing_last_5_years = df_housing.loc[
    df_housing["project_start_date"].between(start_date, latest_date)
].copy()

housing_last_5_years["total_units"] = pd.to_numeric(
    housing_last_5_years["total_units"],
    errors="coerce"
)

monthly_units = (
    housing_last_5_years.groupby("start_month", as_index=False)["total_units"]
    .sum()
    .set_index("start_month")
    .reindex(range(1, 13), fill_value=0)
    .rename_axis("start_month")
    .reset_index()
)

monthly_units["month"] = monthly_units["start_month"].map({
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
})

display(monthly_units)

,start_month,total_units,month
0,1,7299.0,Jan
1,2,7545.0,Feb
2,3,14144.0,Mar
3,4,10225.0,Apr
4,5,15703.0,May
5,6,50846.0,Jun
6,7,9772.0,Jul
7,8,12220.0,Aug
8,9,7672.0,Sep
9,10,8203.0,Oct


In [20]:
fig = px.bar(
    monthly_units,
    x="month",
    y="total_units",
    category_orders={
        "month": ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                  "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    },
    title="Affordable Housing Units by Project Start Month (Last 5 Years)",
    labels={
        "month": "Month",
        "total_units": "Total Housing Units"
    }
)

fig.show()

For the last five years in the data, affordable housing production varies substantially by month. June has the highest total (50,846 units), followed by December (39,702). January is lowest (7,299), with September close behind (7,672).

So, the data suggests production has been concentrated in June and December. These are totals pooled across the five-year period; they don’t show whether the pattern repeats each year. Also, this measures units associated with projects by their start date, not units completed or delivered in that month.

---

### Housing Completion and Traffic Harm

This is an association analysis; it cannot establish that housing completions caused a change.

I will start by aggregating monthly housing completions.

In [21]:
df_housing["building_completion_date"] = pd.to_datetime(
    df_housing["building_completion_date"],
    errors="coerce"
)

df_housing["total_units"] = pd.to_numeric(
    df_housing["total_units"],
    errors="coerce"
)

housing_completed = df_housing.dropna(
    subset=["building_completion_date", "total_units"]
).copy()

housing_completed["month"] = (
    housing_completed["building_completion_date"].dt.to_period("M")
)

monthly_completions = (
    housing_completed.groupby("month")["total_units"]
    .sum()
    .rename("completed_units")
    .to_frame()
)

I will now prepare monthly traffic harm data

In [22]:
df_transport["crash_date"] = pd.to_datetime(
    df_transport["crash_date"],
    errors="coerce"
)

traffic_harm_columns = [
    "total_severe_injuries",
    "total_fatalities"
]

for column in traffic_harm_columns:
    df_transport[column] = pd.to_numeric(
        df_transport[column],
        errors="coerce"
    )

traffic_data = df_transport.dropna(
    subset=["crash_date"]
).copy()

traffic_data["month"] = traffic_data["crash_date"].dt.to_period("M")

monthly_traffic_harm = (
    traffic_data.groupby("month")[traffic_harm_columns]
    .sum()
)

monthly_traffic_harm["traffic_harm"] = (
    monthly_traffic_harm["total_severe_injuries"]
    + monthly_traffic_harm["total_fatalities"]
)

Now, I will join the two sets of data together and visualise the data in a line chart.

In [23]:
monthly_data = monthly_completions.join(
    monthly_traffic_harm[["traffic_harm"]],
    how="inner"
).sort_index()

monthly_data["completed_units_previous_month"] = (
    monthly_data["completed_units"].shift(1)
)

display(monthly_data.head())

,completed_units,traffic_harm,completed_units_previous_month
month,,,
2022-01,601.0,185.0,NaN
2022-02,2133.0,178.0,601.0
2022-03,1654.0,222.0,2133.0
2022-04,9417.0,175.0,1654.0
2022-05,1600.0,285.0,9417.0


In [26]:
monthly_plot_data = monthly_data.reset_index()
monthly_plot_data["month"] = monthly_plot_data["month"].dt.to_timestamp()

fig = px.line(
    monthly_plot_data,
    x="month",
    y=["completed_units", "traffic_harm"],
    facet_row="variable",
    title="Monthly Housing Completions and Traffic Harm",
    labels={
        "month": "Month",
        "value": "Monthly total",
        "variable": "Measure"
    }
)

fig.update_yaxes(matches=None)
fig.show()

It is clear that spikes in monthly housing completions and traffic harm are close to each other. In 2022, the traffic harm spike was directly after the spike in house completions. In 2024 and 2025, there were no house completion spikes.